# 02 训练与导出：Conditional U-Net（Python 3.12+）

输出 checkpoint：`./checkpoints/best.pt`，可直接供 GUI 调用。

In [ ]:
import sys, os
PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
TRAIN_MANIFEST = Path("./manifests/train.json")
VAL_MANIFEST = Path("./manifests/val.json")
assert TRAIN_MANIFEST.exists() and VAL_MANIFEST.exists(), "请先运行 Notebook 01 生成 manifests"
print("TRAIN_MANIFEST:", TRAIN_MANIFEST.resolve())
print("VAL_MANIFEST:", VAL_MANIFEST.resolve())


In [ ]:
import torch
from torch.utils.data import DataLoader

from color_points_dl.dataset import ColorPointsDataset, DataConfig
from color_points_dl.points import PointSamplerConfig
from color_points_dl.unet import UNetColorizer
from color_points_dl.train_loop import TrainConfig, LossConfig, fit
from color_points_dl.utils import seed_everything, get_device


In [ ]:
seed_everything(42)

# 建议先跑通：小一点
CROP_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 0  # macOS / notebook 建议先用 0

point_cfg = PointSamplerConfig(
    mode="random",
    min_points=20,
    max_points=200,
    radius=4,
)

data_cfg = DataConfig(
    crop_size=CROP_SIZE,
    augment_hflip=True,
    point_cfg=point_cfg,
    max_images=None,  # 想先跑通可以设成 2000
)

train_ds = ColorPointsDataset(str(TRAIN_MANIFEST), data_cfg=data_cfg, seed=0, is_train=True)
val_ds = ColorPointsDataset(str(VAL_MANIFEST), data_cfg=data_cfg, seed=1, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("train:", len(train_ds), "val:", len(val_ds))


In [ ]:
device = get_device()
print("device:", device)

model = UNetColorizer(in_channels=4, out_channels=2, base_channels=64)

# None / ""：从头训练
# 例如："./checkpoints/last.pt" 或 "./checkpoints/best.pt"
RESUME_FROM = None
RESET_OPTIMIZER = False  # True: 只加载模型参数；False: 同时恢复优化器（若 checkpoint 中存在）

train_cfg = TrainConfig(
    epochs=20,
    lr=2e-4,
    weight_decay=0.0,
    log_every=50,
    grad_clip=0.0,
    save_dir="./checkpoints",
    best_name="best.pt",
    last_name="last.pt",
    resume_from=RESUME_FROM,
    reset_optimizer=RESET_OPTIMIZER,
)

loss_cfg = LossConfig(lambda_hint=20.0, tv_weight=0.0)

if RESUME_FROM:
    print("将从 checkpoint 继续训练:", RESUME_FROM)
else:
    print("将从头开始训练")


In [ ]:
history = fit(model, train_loader, val_loader, device, train_cfg, loss_cfg)
history


## 训练曲线

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(True)
plt.legend()
plt.show()


## 可视化：Input view / Pred / GT（验证集）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

from color_points_dl.color_utils import lab_norm_to_bgr

@torch.no_grad()
def visualize(n=4):
    model.eval()
    batch = next(iter(val_loader))
    x = batch["x"][:n].to(device)
    y = batch["y"][:n].numpy()
    L = batch["L"][:n].numpy()
    mask = batch["mask"][:n].numpy()

    pred = model(x).cpu().numpy()
    hint_ab = batch["x"][:n, 1:3].numpy()
    m = mask.astype(bool)
    pred[m.repeat(2, axis=1)] = hint_ab[m.repeat(2, axis=1)]

    plt.figure(figsize=(12, 3*n))
    for i in range(n):
        bgr_gt = lab_norm_to_bgr(L[i], y[i])
        bgr_in = lab_norm_to_bgr(L[i], hint_ab[i])
        bgr_pr = lab_norm_to_bgr(L[i], pred[i])

        rgb_gt = cv2.cvtColor(bgr_gt, cv2.COLOR_BGR2RGB)
        rgb_in = cv2.cvtColor(bgr_in, cv2.COLOR_BGR2RGB)
        rgb_pr = cv2.cvtColor(bgr_pr, cv2.COLOR_BGR2RGB)

        plt.subplot(n,3,i*3+1); plt.imshow(rgb_gt); plt.title("GT"); plt.axis("off")
        plt.subplot(n,3,i*3+2); plt.imshow(rgb_in); plt.title("Input view"); plt.axis("off")
        plt.subplot(n,3,i*3+3); plt.imshow(rgb_pr); plt.title("Pred"); plt.axis("off")
    plt.tight_layout()
    plt.show()

visualize(4)


## GUI 调用示例

```python
from color_points_dl.infer import Colorizer
colorizer = Colorizer.from_checkpoint('./checkpoints/best.pt')
out_bgr = colorizer.colorize_bgr(img_bgr, points=[(x,y), ...], radius=4)
```
